In [1]:
!pip install phe

  Using cached phe-1.5.0-py2.py3-none-any.whl.metadata (3.8 kB)
Using cached phe-1.5.0-py2.py3-none-any.whl (53 kB)


In [2]:
!pip install haversine

  Using cached haversine-2.9.0-py2.py3-none-any.whl.metadata (5.8 kB)
Using cached haversine-2.9.0-py2.py3-none-any.whl (7.7 kB)


In [3]:
import math
from phe import paillier
from haversine import haversine

# Step 1: Key Generation (Alice)
public_key, private_key = paillier.generate_paillier_keypair()

# Example coordinates in degrees
lat_A = 50.379320  # Alice's latitude
lon_A = -4.131244  # Alice's longitude
lat_B = 50.381813  # Bob's latitude
lon_B = -4.127100  # Bob's longitude

# Define the geofence
geofence_radius = 0.3  # Radius in kilometers

# Convert degrees to radians
latA = math.radians(lat_A)
lonA = math.radians(lon_A)
latB = math.radians(lat_B)
lonB = math.radians(lon_B)

# Compute the trigonometric values as per the protocol
alpha = math.cos(latA / 2)
beta = math.sin(latB / 2)
gamma = math.sin(latA / 2)
delta = math.cos(latB / 2)
zeta = math.cos(latA)
eta = math.cos(latB)
theta = math.sin(lonA / 2)
lambda_ = math.cos(lonB / 2)
mu = math.cos(lonA / 2)
nu = math.sin(lonB / 2)

# Step 2: Alice computes encrypted values and sends them to Bob
alpha_squared = alpha**2
neg_two_alpha_gamma = -2 * alpha * gamma
gamma_squared = gamma**2
zeta_eta_theta_lambda_squared = zeta * eta * (theta**2) * (lambda_**2)
neg_two_zeta_eta_theta_lambda = -2 * zeta * eta * theta * lambda_
zeta_eta = zeta * eta

# Encrypt the values
enc_alpha_squared = public_key.encrypt(alpha_squared)
enc_neg_two_alpha_gamma = public_key.encrypt(neg_two_alpha_gamma)
enc_gamma_squared = public_key.encrypt(gamma_squared)
enc_zeta_eta_theta_lambda_squared = public_key.encrypt(zeta_eta_theta_lambda_squared)
enc_neg_two_zeta_eta_theta_lambda = public_key.encrypt(neg_two_zeta_eta_theta_lambda)
enc_zeta_eta = public_key.encrypt(zeta_eta)

# Alice sends encrypted values to Bob
alice_data = {
    "enc_alpha_squared": enc_alpha_squared,
    "enc_neg_two_alpha_gamma": enc_neg_two_alpha_gamma,
    "enc_gamma_squared": enc_gamma_squared,
    "enc_zeta_eta_theta_lambda_squared": enc_zeta_eta_theta_lambda_squared,
    "enc_neg_two_zeta_eta_theta_lambda": enc_neg_two_zeta_eta_theta_lambda,
    "enc_zeta_eta": enc_zeta_eta,
}

# Step 3: Bob computes JaK using homomorphic operations
beta_squared = beta**2
delta_squared = delta**2
mu_nu = mu * nu
mu_squared_nu_squared = mu**2 * nu**2

enc_a = (
    alice_data["enc_alpha_squared"] * beta_squared
    + alice_data["enc_neg_two_alpha_gamma"] * (beta * delta)
    + alice_data["enc_gamma_squared"] * delta_squared
    + alice_data["enc_zeta_eta_theta_lambda_squared"]
    + alice_data["enc_neg_two_zeta_eta_theta_lambda"] * mu_nu
    + alice_data["enc_zeta_eta"] * mu_squared_nu_squared
)

# Bob sends enc_a back to Alice
# Step 4: Alice decrypts JaK and computes the distance
a = private_key.decrypt(enc_a)

# Earth's radius in kilometers
R = 6371.0

# Compute haversine distance
distance = 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Determine if Alice's point is inside or outside the geofence
if distance <= geofence_radius:
    print(f"ALice's location is INSIDE the geofence (distance: {distance:.2f} km).")
else:
    print(f"Alice's location is OUTSIDE the geofence (distance: {distance:.2f} km).")

# Output the computed distance for verification
print(f"Privacy-preserving haversine distance: {distance:.2f} km")

# Validate using standard haversine library
haversine_distance = haversine((lat_A, lon_A), (lat_B, lon_B))
print(f"Distance between Alice and Bob (using haversine library): {haversine_distance:.2f} km")

print(enc_a.ciphertext())

Alice's location is OUTSIDE the geofence (distance: 0.40 km).
Privacy-preserving haversine distance: 0.40 km
Distance between Alice and Bob (using haversine library): 0.40 km
288768070540853901360047610983782863208854595663608161399650672742761585677731383368500569774997739337142953413981211745972967476456088541419804032285298286330085634257800960222362259268098602806947949604365844713492962948188293037760928917739814675676035709518219363013746729617534548160002341780945269219379599211564584059544791692240543460412294822486443795746739938063798203846580607062710742637259512170841305684961749086572083706959378779110536688826358915011880973772033547706970553503075371675348605668716442940449728168128217760048449899842663749652137095753113058730638875111291998055321298373776223399255978938943278418032563276213543541127582812879633266270136097824384529832675828221523957381397413915353578915743618375424973990762632533615756160505756561366714595018415548479660096625290717461986866961024016412